# Portfolio Bot — Liquidation Percentage Optimisation

Grid search over sell ratios (10%–90%) to find optimal balance.

Fixed: $20/mo, BTC 1%+2%->50%, BNB 1%+1%->40%

In [ ]:
import pandas as pd, requests, time, numpy as np, matplotlib.pyplot as plt
from datetime import datetime, timezone
pd.set_option('display.max_columns', None); pd.set_option('display.width', 200)
pd.set_option('display.float_format', lambda x: '%.4f' % x)

In [ ]:
def fetch_klines(symbol):
    url = 'https://api.binance.com/api/v3/klines'
    all_k = []; st = 0
    while True:
        p = {'symbol': symbol, 'interval': '1M', 'startTime': st, 'limit': 500}
        d = requests.get(url, params=p).json()
        if not d: break
        all_k.extend(d)
        if len(d) < 500: break
        st = d[-1][0] + 1; time.sleep(0.1)
    return all_k
cols = ['timestamp','open','high','low','close','volume','close_time','quote_vol','trades','taker_buy_base','taker_buy_quote','ignore']
btc_k = fetch_klines('BTCUSDT'); bnb_k = fetch_klines('BNBUSDT')
btc = pd.DataFrame(btc_k, columns=cols); btc['date'] = pd.to_datetime(btc['timestamp'], unit='ms', utc=True)
bnb = pd.DataFrame(bnb_k, columns=cols); bnb['date'] = pd.to_datetime(bnb['timestamp'], unit='ms', utc=True)
for c in ['open','high','low','close','volume']: btc[c] = btc[c].astype(float); bnb[c] = bnb[c].astype(float)
btc = btc[['date','open','high','low','close','volume']].copy().sort_values('date').reset_index(drop=True)
bnb = bnb[['date','open','high','low','close','volume']].copy().sort_values('date').reset_index(drop=True)
start = max(btc['date'].min(), bnb['date'].min())
end = min(btc['date'].max(), bnb['date'].max())
btc = btc[(btc['date'] >= start) & (btc['date'] <= end)].reset_index(drop=True)
bnb = bnb[(bnb['date'] >= start) & (bnb['date'] <= end)].reset_index(drop=True)
print(f'Period: {btc["date"].min().strftime("%Y-%m")} -> {btc["date"].max().strftime("%Y-%m")} ({len(btc)} months)')

In [ ]:
def run_portfolio(sell_pct):
    sell_ratio = sell_pct / 100.0
    usdt = 0.0; injected = 0.0
    btc_h = 0.0; btc_hi = 0.0; btc_rp = 1.0; btc_inv = 0.0
    bnb_h = 0.0; bnb_hi = 0.0; bnb_rp = 1.0; bnb_inv = 0.0
    vals = []
    for i in range(len(btc)):
        br = btc.iloc[i]; nr = bnb.iloc[i]
        bc = br['close']; nc = nr['close']
        usdt += 20.0; injected += 20.0
        if bc < br['open']:
            ext = usdt * (btc_rp / 100.0) if usdt > 0 else 0.0
            spend = min(10.0 + ext, usdt)
            btc_h += spend / bc; btc_inv += 10.0; usdt -= spend
        if nc < nr['open']:
            ext = usdt * (bnb_rp / 100.0) if usdt > 0 else 0.0
            spend = min(10.0 + ext, usdt)
            bnb_h += spend / nc; bnb_inv += 10.0; usdt -= spend
        bp = btc_hi
        if bc > btc_hi: btc_hi = bc
        if btc_h > 0.000001 and bc > bp:
            sell = btc_h * sell_ratio
            usdt += sell * bc; btc_h -= sell; btc_rp = 1.0
        elif btc_rp < 50: btc_rp = min(50, btc_rp + 2)
        np_ = bnb_hi
        if nc > bnb_hi: bnb_hi = nc
        if bnb_h > 0.000001 and nc > np_:
            sell = bnb_h * sell_ratio
            usdt += sell * nc; bnb_h -= sell; bnb_rp = 1.0
        elif bnb_rp < 40: bnb_rp = min(40, bnb_rp + 1)
        vals.append(btc_h * bc + bnb_h * nc + usdt)
    fv = vals[-1]
    ret = (fv / injected - 1) * 100
    eq = pd.Series(vals)
    dd = ((eq.cummax() - eq) / eq.cummax()).max() * 100
    m = eq.pct_change().dropna()
    m = m[m.index >= 12]
    sharpe = (m.mean() / m.std()) * np.sqrt(12) if m.std() > 0 else 0
    pos = m[m > 0].sum(); neg = m[m < 0].abs().sum()
    pf = pos / neg if neg > 0 else float('inf')
    ann = (1 + ret/100) ** (12/len(btc)) - 1
    calmar = ann / (dd/100) if dd > 0 else 0
    return {'sell': sell_pct, 'return': ret, 'dd': dd, 'sharpe': sharpe,
            'calmar': calmar, 'pf': pf, 'final': fv, 'injected': injected,
            'reserve': usdt}
sell_ratios = list(range(10, 95, 5))
results = []
for s in sell_ratios:
    r = run_portfolio(s)
    results.append(r)
    print(f"Sell {s:>2d}%  R={r['return']:7.1f}%  DD={r['dd']:5.1f}%  Sharpe={r['sharpe']:.2f}  Calmar={r['calmar']:.2f}  Final=${r['final']:.0f}")
opt = pd.DataFrame(results)
print()
print('Done.')

In [ ]:
print('=== TOP 5 BY CALMAR ===')
print(opt.nlargest(5, 'calmar')[['sell','return','dd','sharpe','calmar','pf','final']].to_string(index=False))
print()
print('=== TOP 5 BY RETURN ===')
print(opt.nlargest(5, 'return')[['sell','return','dd','sharpe','calmar','pf','final']].to_string(index=False))
print()
print('=== TOP 5 BY SHARPE ===')
print(opt.nlargest(5, 'sharpe')[['sell','return','dd','sharpe','calmar','pf','final']].to_string(index=False))
print()
print('=== BEST WITH DD < 30% ===')
dd30 = opt[opt['dd'] < 30]
if len(dd30) > 0:
    print(dd30.nlargest(5, 'calmar')[['sell','return','dd','sharpe','calmar','pf','final']].to_string(index=False))
else:
    print('None')
print()
print('=== BEST WITH DD < 40% ===')
dd40 = opt[opt['dd'] < 40]
if len(dd40) > 0:
    print(dd40.nlargest(5, 'calmar')[['sell','return','dd','sharpe','calmar','pf','final']].to_string(index=False))
else:
    print('None')

In [ ]:
# Chart: tradeoff curves
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
sells = [r['sell'] for r in results]
ax = axes[0, 0]
ax.plot(sells, [r['return'] for r in results], 'o-', color='#2563eb', linewidth=2, markersize=8)
ax.set_xlabel('Sell % at ATH'); ax.set_ylabel('Return (%)'); ax.set_title('Return vs Sell %'); ax.grid(alpha=0.3)
ax = axes[0, 1]
ax.plot(sells, [r['dd'] for r in results], 'o-', color='#dc2626', linewidth=2, markersize=8)
ax.set_xlabel('Sell % at ATH'); ax.set_ylabel('Max DD (%)'); ax.set_title('Drawdown vs Sell %'); ax.grid(alpha=0.3)
ax = axes[1, 0]
ax.plot(sells, [r['sharpe'] for r in results], 'o-', color='#16a34a', linewidth=2, markersize=8)
ax.set_xlabel('Sell % at ATH'); ax.set_ylabel('Sharpe Ratio'); ax.set_title('Sharpe vs Sell %'); ax.grid(alpha=0.3)
ax = axes[1, 1]
ax.plot(sells, [r['calmar'] for r in results], 'o-', color='#8b5cf6', linewidth=2, markersize=8)
ax.set_xlabel('Sell % at ATH'); ax.set_ylabel('Calmar Ratio'); ax.set_title('Calmar vs Sell %'); ax.grid(alpha=0.3)
plt.suptitle('Portfolio Bot - Liquidation % Optimisation', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('dca-bt-buy-bot-portfolio-opt.png', dpi=150, bbox_inches='tight')
print('Chart saved as dca-bt-buy-bot-portfolio-opt.png')
plt.show()

In [ ]:
# Detailed comparison table
print(f"{'Sell%':>6s} {'Return':>8s} {'DD':>6s} {'Sharpe':>7s} {'Calmar':>7s} {'PF':>7s} {'Final':>10s} {'Reserve':>9s}")
print('-'*66)
for r in results:
    print(f"{r['sell']:>5d}% {r['return']:>7.1f}% {r['dd']:>5.1f}% {r['sharpe']:>7.4f} {r['calmar']:>7.4f} {r['pf']:>7.2f} ${r['final']:>8.0f} ${r['reserve']:>8.0f}")